# Fish Speech (S2-Pro) WebUI on Google Colab

This notebook will launch the Fish Speech WebUI, giving you an experience similar to the official Fish Audio website.

### Instructions:
1. **Change Runtime Type**: Go to `Runtime` -> `Change runtime type` -> select **T4 GPU** (or a better GPU like A100/V100).
2. **Get Ngrok Token**: Go to [ngrok.com](https://ngrok.com/), sign up/log in, and get your authtoken from the dashboard.
3. **Run All**: Go to `Runtime` -> `Run all` (or press `Ctrl+F9`). 
4. When prompted, enter your ngrok authtoken in the text box.
5. Wait for the installation and model download to finish (~5-10 minutes).
6. Click the `ngrok.io` link printed at the bottom to access the WebUI!

In [ ]:
# Check GPU Availability
!nvidia-smi

In [ ]:
# Clone the repository and install dependencies
import os

!git clone https://github.com/fishaudio/fish-speech.git
os.chdir("fish-speech")

!pip install -q uv
!apt-get update -q && apt-get install -y -q portaudio19-dev
!uv pip install --system -e ".[cu126]" || (sed -i '/pyaudio/d' pyproject.toml && uv pip install --system -e ".[cu126]")
!pip install torchvision==0.23.0+cu126 --index-url https://download.pytorch.org/whl/cu126
!pip install -e audio-preprocess

print("Dependencies installed successfully!")

In [ ]:
# Download the S2-Pro Model
from huggingface_hub import snapshot_download

print("Downloading Fish Speech S2-Pro model (this may take a few minutes)...")
snapshot_download("fishaudio/s2-pro", local_dir="checkpoints/s2-pro")
print("Model download complete!")

In [ ]:
%%writefile standalone_inference.py
import os
import subprocess
import argparse
from pathlib import Path

def run_cmd(cmd, env=None):
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, check=True, env=env)

def main():
    parser = argparse.ArgumentParser(description="End-to-End Inference matching FishAudio Website Quality")
    parser.add_argument("--text", type=str, required=True, help="Text to synthesize")
    parser.add_argument("--reference-audio", type=str, required=True, help="Path to reference audio")
    parser.add_argument("--prompt-text", type=str, required=True, help="Transcript of the reference audio")
    parser.add_argument("--output", type=str, default="output.wav", help="Path to output audio")
    parser.add_argument("--checkpoint-path", type=str, default="checkpoints/s2-pro", help="Path to the model checkpoint")
    args = parser.parse_args()

    tmp_dir = Path("tmp_processing")
    tmp_dir.mkdir(exist_ok=True)
    
    ref_audio_path = Path(args.reference_audio)
    
    # 1. Vocal Separation (Audio Preprocessing)
    print("\n--- Step 1: Separating Vocals ---")
    sep_dir = tmp_dir / "separated"
    sep_dir.mkdir(exist_ok=True)
    run_cmd([
        "python", "-m", "fish_audio_preprocess.cli.separate_audio",
        "--input_dir", str(ref_audio_path.parent),
        "--output_dir", str(sep_dir)
    ])
    
    sep_audio_path = sep_dir / ref_audio_path.name
    if not sep_audio_path.exists():
        sep_audio_path = ref_audio_path

    # 2. Loudness Normalization (Audio Preprocessing)
    print("\n--- Step 2: Normalizing Loudness ---")
    norm_dir = tmp_dir / "normalized"
    norm_dir.mkdir(exist_ok=True)
    run_cmd([
        "python", "-m", "fish_audio_preprocess.cli.loudness_norm",
        "--input_dir", str(sep_dir),
        "--output_dir", str(norm_dir),
        "--loudness", "-23"
    ])
    
    norm_audio_path = norm_dir / ref_audio_path.name
    if not norm_audio_path.exists():
        norm_audio_path = sep_audio_path

    # 3. Generate Prompt Tokens (fish-speech)
    print("\n--- Step 3: Generating Prompt Tokens ---")
    run_cmd([
        "python", "fish_speech/models/dac/inference.py",
        "-i", str(norm_audio_path),
        "--checkpoint-path", f"{args.checkpoint_path}/codec.pth"
    ])
    
    prompt_tokens_path = "fake.npy"

    # 4. Generate Semantic Tokens from Text (fish-speech)
    print("\n--- Step 4: Generating Semantic Tokens ---")
    run_cmd([
        "python", "fish_speech/models/text2semantic/inference.py",
        "--text", args.text,
        "--prompt-text", args.prompt_text,
        "--prompt-tokens", prompt_tokens_path,
        "--checkpoint-path", args.checkpoint_path,
        "--num-samples", "1"
    ])
    
    semantic_tokens_path = "codes_0.npy"

    # 5. Decode to Waveform (Vocoder)
    print("\n--- Step 5: Decoding to Waveform using Vocoder ---")
    run_cmd([
        "python", "fish_speech/models/dac/inference.py",
        "-i", semantic_tokens_path,
        "--checkpoint-path", f"{args.checkpoint_path}/codec.pth"
    ])
    
    generated_audio_path = Path("fake.wav")
    if generated_audio_path.exists():
        generated_audio_path.rename(args.output)
        print(f"\n✅ Success! Audio saved to {args.output}")
    else:
        print("\n❌ Error: Output audio not generated.")

if __name__ == "__main__":
    main()


### Standalone Quality Inference
Run the cell below with your own reference audio to test the integrated preprocessing pipeline (Separation + Normalization) for maximum quality!

In [ ]:
# Example usage:
# Upload 'my_reference.wav' to Colab first, then run this:
# !python standalone_inference.py --text "Hello, world!" --reference-audio "my_reference.wav" --prompt-text "Transcript of reference" --output "final.wav"

from IPython.display import Audio
# Audio("final.wav") # Uncomment to play the generated audio

In [ ]:
# Configure WebUI and Launch with Ngrok
import subprocess
import time
import getpass

# 1. Setup Ngrok
!pip install -q pyngrok
from pyngrok import ngrok

print("Please enter your Ngrok Authtoken (get it from https://dashboard.ngrok.com/get-started/your-authtoken):")
ngrok_token = getpass.getpass("Ngrok Token: ")
ngrok.set_auth_token(ngrok_token)

# 2. Modify run_webui.py to expose the gradio server
!sed -i 's/app.launch()/app.launch(server_name="0.0.0.0", share=False)/' tools/run_webui.py

# 3. Open ngrok tunnel to port 7860
public_url = ngrok.connect(7860).public_url
print(f"\n\033[1m\033[92m=== WEB UI IS AVAILABLE AT: {public_url} ===\033[0m\n")

# 4. Launch the Web UI
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Starting WebUI... (Keep this cell running to use the app)")
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python tools/run_webui.py \
    --llama-checkpoint-path checkpoints/s2-pro \
    --decoder-checkpoint-path checkpoints/s2-pro/codec.pth \
    --half
